# Phase 1: LoRA Fine-Tuning Trump Speech Model

Fine-tunes Llama-3-8B or Mistral-7B on Trump's speech corpus using LoRA via Unsloth.

**Requirements**: Google Colab Pro (A100 or V100 GPU)

**Input**: `trump_speeches.jsonl` from your bot's data exporter

**Output**: LoRA adapter weights saved to Google Drive

## 0. Auto-Pipeline Trigger Check

When this notebook is run on Colab's built-in scheduler, this cell checks
Google Drive for a `training_trigger.json` file written by your local
FastAPI server. If no trigger is found, the notebook exits early to save
GPU credits. If a trigger is found, training proceeds normally.

In [ ]:
import os
import json
from datetime import datetime

# Mount Drive first (needed to check for trigger)
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/trump_trading_bot'
TRIGGER_DIR = os.path.join(PROJECT_DIR, 'triggers')
TRIGGER_PATH = os.path.join(TRIGGER_DIR, 'training_trigger.json')

# Check for automated trigger
AUTO_TRIGGERED = False
trigger_data = {}

if os.path.exists(TRIGGER_PATH):
    with open(TRIGGER_PATH) as f:
        trigger_data = json.load(f)
    AUTO_TRIGGERED = True
    print(f"✅ Training trigger found!")
    print(f"   Type: {trigger_data.get('trigger_type', 'unknown')}")
    print(f"   Triggered at: {trigger_data.get('triggered_at', 'unknown')}")
    print(f"   Data: {json.dumps(trigger_data.get('request', {}), indent=2)}")
else:
    # If running manually, proceed. If on scheduler, exit early.
    import sys
    is_scheduled = os.getenv('COLAB_SCHEDULED_RUN', '') == '1'
    if is_scheduled:
        print("⏭️  No training trigger found. Scheduled run — exiting early to save GPU credits.")
        sys.exit(0)
    else:
        print("ℹ️  No trigger file found — running manually. Proceeding with training.")
        AUTO_TRIGGERED = False

In [ ]:
# Install dependencies
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers sentencepiece protobuf

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set paths
import os
PROJECT_DIR = '/content/drive/MyDrive/trump_trading_bot'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'Project dir: {PROJECT_DIR}')
print(f'Upload your trump_speeches.jsonl to: {DATA_DIR}')

In [ ]:
# Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Load Base Model with Unsloth

Unsloth patches the model for 2x faster training and 60% less VRAM.
Choose your base model below.

In [ ]:
from unsloth import FastLanguageModel

# === CHOOSE YOUR BASE MODEL ===
# Option A: Llama 3 8B (recommended - best quality)
BASE_MODEL = 'unsloth/Meta-Llama-3.1-8B'

# Option B: Mistral 7B (slightly faster, also excellent)
# BASE_MODEL = 'unsloth/mistral-7b-v0.3'

# Option C: Smaller model if VRAM is tight
# BASE_MODEL = 'unsloth/Phi-3.5-mini-instruct'

MAX_SEQ_LENGTH = 2048
DTYPE = None  # auto-detect (float16 for V100, bfloat16 for A100)
LOAD_IN_4BIT = True  # 4-bit quantization to fit in VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f'Loaded {BASE_MODEL}')
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,                    # LoRA rank - higher = more capacity, more VRAM
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',  # attention
        'gate_proj', 'up_proj', 'down_proj',       # MLP
    ],
    lora_alpha=32,           # scaling factor
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # saves 60% VRAM
    random_state=42,
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})')

## 2. Prepare Training Dataset

In [ ]:
from datasets import load_dataset

# Load the exported speech corpus
DATASET_PATH = os.path.join(DATA_DIR, 'trump_speeches.jsonl')

if not os.path.exists(DATASET_PATH):
    print(f'ERROR: Upload trump_speeches.jsonl to {DATA_DIR}')
    print('Run this on your local machine first:')
    print('  python -c "from src.ml.data_exporter import DataExporter; DataExporter().export_training_corpus()"')
    print(f'  Then upload data/exports/trump_speeches.jsonl to Google Drive at {DATA_DIR}')
else:
    dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
    print(f'Loaded {len(dataset)} training examples')
    print(f'Sample: {dataset[0]["text"][:200]}...')

In [ ]:
# Format for causal language modeling
# The model learns to predict the next token in Trump's speech patterns

EOS_TOKEN = tokenizer.eos_token

def format_for_training(examples):
    """Format each speech chunk for causal LM training."""
    texts = []
    for text in examples['text']:
        # Add EOS token so the model learns where speeches end
        texts.append(text + EOS_TOKEN)
    return {'text': texts}

dataset = dataset.map(format_for_training, batched=True)
print(f'Dataset ready: {len(dataset)} examples')

## 3. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,  # packs short sequences together for efficiency
    args=TrainingArguments(
        output_dir=os.path.join(MODEL_DIR, 'checkpoints'),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,  # effective batch size = 16
        warmup_steps=50,
        num_train_epochs=3,             # 3 epochs is usually good
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100,
        save_total_limit=3,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        report_to='none',
    ),
)

print('Starting training...')
stats = trainer.train()
print(f'Training complete! Loss: {stats.training_loss:.4f}')

In [ ]:
# Quick test: generate a sample
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    'We are going to make America',
    return_tensors='pt'
).to('cuda')

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
)

print('=== Sample Generation ===')
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 4. Save the LoRA Adapter

Only saves the adapter weights (~100-200MB), not the full model.

In [ ]:
# Save LoRA adapter to Google Drive
ADAPTER_PATH = os.path.join(MODEL_DIR, 'trump_lora_adapter')

# Save adapter weights (safetensors format) and tokenizer
model.save_pretrained(ADAPTER_PATH, safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_PATH)

# Ensure adapter_config.json has base_model_name_or_path set
# (some Unsloth/PEFT versions omit this, which breaks loading)
adapter_config_path = os.path.join(ADAPTER_PATH, 'adapter_config.json')
with open(adapter_config_path) as f:
    adapter_config = json.load(f)
if not adapter_config.get('base_model_name_or_path'):
    adapter_config['base_model_name_or_path'] = BASE_MODEL
    with open(adapter_config_path, 'w') as f:
        json.dump(adapter_config, f, indent=2)
    print(f'Patched adapter_config.json with base_model_name_or_path = {BASE_MODEL}')

# Verify the save has the right files
expected_files = ['adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json']
for fname in expected_files:
    fpath = os.path.join(ADAPTER_PATH, fname)
    exists = '✅' if os.path.exists(fpath) else '❌ MISSING'
    print(f'  {exists} {fname}')

# Check size
import subprocess
size = subprocess.check_output(['du', '-sh', ADAPTER_PATH]).decode().split()[0]
print(f'\nAdapter saved to: {ADAPTER_PATH}')
print(f'Adapter size: {size}')

In [ ]:
# Optional: Save as merged GGUF for local inference with llama.cpp
# This is useful if you want to run inference locally without Colab

SAVE_GGUF = False  # Set to True if you want a GGUF export

if SAVE_GGUF:
    GGUF_PATH = os.path.join(MODEL_DIR, 'trump_model_q4')
    model.save_pretrained_gguf(
        GGUF_PATH,
        tokenizer,
        quantization_method='q4_k_m',
    )
    print(f'GGUF model saved to: {GGUF_PATH}')

## 5. Signal Completion (Auto-Pipeline)

If this was an auto-triggered run, write a completion signal to Drive
and clean up the trigger file so the local server knows training is done.

In [ ]:
if AUTO_TRIGGERED:
    # Write completion signal
    completion_data = {
        'completed_at': datetime.utcnow().isoformat(),
        'status': 'success',
        'phase': 'fine_tuning',
        'model_path': ADAPTER_PATH,
        'training_loss': stats.training_loss if 'stats' in dir() else None,
        'trigger_data': trigger_data,
    }

    os.makedirs(TRIGGER_DIR, exist_ok=True)
    completion_path = os.path.join(TRIGGER_DIR, 'training_complete.json')
    with open(completion_path, 'w') as f:
        json.dump(completion_data, f, indent=2)

    # Clean up trigger file
    if os.path.exists(TRIGGER_PATH):
        os.remove(TRIGGER_PATH)

    print(f"✅ Completion signal written to: {completion_path}")
    print(f"   Your local server will pick this up on its next poll cycle.")
else:
    print("ℹ️  Manual run — no completion signal written.")
    print("   To import predictions, run the Monte Carlo notebook next.")